# CivicLens: 02. Baseline Classification Models
This notebook trains and benchmarks lexical TF-IDF baselines (Logistic Regression and Calibrated Linear SVM) on complaint categorization.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
from src.preprocessing import validate_complaints_data, grouped_train_test_split
from src.classification import TfidfBaselineClassifier
from src.evaluation import evaluate_classification_predictions
from src.visualization import plot_confusion_matrix

## 1. Load Data and Perform Grouped Split

In [ ]:
df = validate_complaints_data(pd.read_csv('../data/sample_complaints.csv'))
train_df, test_df = grouped_train_test_split(df, group_col='incident_group_id', test_size=0.25, random_state=42)
print(f'Train size: {len(train_df)}, Test size: {len(test_df)}')

## 2. Train Word TF-IDF + Logistic Regression

In [ ]:
clf_lr = TfidfBaselineClassifier(classifier_type='logistic_regression', ngram_range=(1, 2))
clf_lr.fit(train_df['text'].tolist(), train_df['category'].tolist())
preds_lr = clf_lr.predict(test_df['text'].tolist())
metrics_lr = evaluate_classification_predictions(test_df['category'].values, preds_lr)
print('Accuracy:', metrics_lr['accuracy'])
print('Macro F1:', metrics_lr['macro_f1'])

## 3. Train Character N-gram TF-IDF + Calibrated Linear SVM

In [ ]:
clf_svm = TfidfBaselineClassifier(classifier_type='linear_svm', analyzer='char_wb', ngram_range=(3, 5))
clf_svm.fit(train_df['text'].tolist(), train_df['category'].tolist())
preds_svm = clf_svm.predict(test_df['text'].tolist())
metrics_svm = evaluate_classification_predictions(test_df['category'].values, preds_svm)
print('Accuracy:', metrics_svm['accuracy'])
print('Macro F1:', metrics_svm['macro_f1'])

## 4. Confusion Matrix

In [ ]:
fig = plot_confusion_matrix(metrics_svm['confusion_matrix'], labels=metrics_svm['labels'], title='TF-IDF Char SVM Confusion Matrix')
plt.show()